# 01 — Manual GBD file download according to OECD country criterion

## Objective

This notebook documents:

1. The manual download procedure from the Global Burden of Disease (GBD) portal.
2. The definition of the analytical cohort: OECD member countries.
3. The construction of the OECD country list used in the analysis.
4. The export of that list to `data/3_processed/oecd_members.csv`.

## Methodological decision

The primary analytical cohort is restricted to OECD member countries.

Rationale:

- Higher institutional and health system comparability.
- Reduced structural heterogeneity relative to global or SDI-based groupings.
- Clear and externally defined membership criteria.

The SDI-based approach is retained as a separate sensitivity analysis outside the main repository.

## OECD member countries source

The official list of OECD member countries is published by the OECD:

https://www.oecd.org/about/members-and-partners/

Several programmatic extraction approaches were evaluated (web scraping, OECD SDMX API, and knowledge graph queries). However:

- OECD web pages are protected by Cloudflare, preventing automated scraping.
- The OECD SDMX API provides country code lists but **does not encode OECD membership**.
- External knowledge graphs (e.g., Wikidata, DBpedia) require prior knowledge of specific data model properties.

Given that the OECD membership list is **short (38 countries), stable, and publicly accessible**, the most robust and transparent approach is to:

1. Verify the list directly from the OECD website.
2. Store it as a versioned dataset within the repository.

The canonical list used in this project is therefore stored as:

`data/3_processed/oecd_members.csv`

In [19]:
from src.paths import RAW_DIR, INTERIM_DIR, PROCESSED_DIR, ensure_data_dirs

ensure_data_dirs()

print("RAW_DIR:", RAW_DIR)
print("PROCESSED_DIR:", PROCESSED_DIR)

RAW_DIR: C:\Users\samai\OneDrive\Escritorio\PROYECTO AUTISMO\autism-diagnosis-gender-gap\data\1_raw
PROCESSED_DIR: C:\Users\samai\OneDrive\Escritorio\PROYECTO AUTISMO\autism-diagnosis-gender-gap\data\3_processed


In [20]:
# --- Create canonical OECD members dataset ---

import pandas as pd

oecd_members_list = [
    "Australia","Austria","Belgium","Canada","Chile","Colombia","Costa Rica",
    "Czechia","Denmark","Estonia","Finland","France","Germany","Greece",
    "Hungary","Iceland","Ireland","Israel","Italy","Japan","South Korea",
    "Latvia","Lithuania","Luxembourg","Mexico","Netherlands","New Zealand",
    "Norway","Poland","Portugal","Slovakia","Slovenia","Spain","Sweden",
    "Switzerland","Turkey","United Kingdom","United States"
]

oecd_members = pd.DataFrame({"country": oecd_members_list})

oecd_members.to_csv(PROCESSED_DIR / "oecd_members.csv", index=False)

print("Saved:", PROCESSED_DIR / "oecd_members.csv")

Saved: C:\Users\samai\OneDrive\Escritorio\PROYECTO AUTISMO\autism-diagnosis-gender-gap\data\3_processed\oecd_members.csv


In [21]:
# --- Load and validate OECD members dataset ---

import pandas as pd

oecd_members = pd.read_csv(PROCESSED_DIR / "oecd_members.csv")

print("Rows:", len(oecd_members))
print("Columns:", list(oecd_members.columns))

oecd_members.head()

Rows: 38
Columns: ['country']


,country
0,Australia
1,Austria
2,Belgium
3,Canada
4,Chile


In [22]:
# --- Load GBD autism prevalence dataset (OECD members selection) ---

import pandas as pd

gbd_file = RAW_DIR / "ihme(gbd)_asd_prevalence_oecd_1990_2023.csv"

gbd_raw = pd.read_csv(gbd_file)

print("Rows:", len(gbd_raw))
print("Columns:", list(gbd_raw.columns))

gbd_raw.head()

Rows: 38760
Columns: ['population_group_id', 'population_group_name', 'measure_id', 'measure_name', 'location_id', 'location_name', 'sex_id', 'sex_name', 'age_id', 'age_name', 'cause_id', 'cause_name', 'metric_id', 'metric_name', 'year', 'val', 'upper', 'lower']


,population_group_id,population_group_name,measure_id,measure_name,location_id,location_name,sex_id,sex_name,age_id,age_name,cause_id,cause_name,metric_id,metric_name,year,val,upper,lower
0,1,Toda la población,5,Prevalencia,55,Eslovenia,1,Hombre,1,Menores de 5 años,575,Trastornos del espectro autista,3,Tasa,1991,939.037586,1855.692963,459.595189
1,1,Toda la población,5,Prevalencia,55,Eslovenia,2,Mujer,1,Menores de 5 años,575,Trastornos del espectro autista,3,Tasa,1991,363.753608,718.021074,177.685811
2,1,Toda la población,5,Prevalencia,55,Eslovenia,1,Hombre,6,5-9 años,575,Trastornos del espectro autista,3,Tasa,1991,920.239063,1984.293942,435.700531
3,1,Toda la población,5,Prevalencia,55,Eslovenia,2,Mujer,6,5-9 años,575,Trastornos del espectro autista,3,Tasa,1991,356.527957,772.392177,168.000119
4,1,Toda la población,5,Prevalencia,55,Eslovenia,1,Hombre,7,10-14 años,575,Trastornos del espectro autista,3,Tasa,1991,912.201931,1979.073395,358.173398


In [24]:
# --- Basic structural validation of GBD dataset ---

key_cols = ["location_name", "sex_name", "age_name", "year", "val"]

print("Countries:", gbd_raw["location_name"].nunique())
print("Sex categories:", sorted(gbd_raw["sex_name"].unique().tolist()))
print("Years:", gbd_raw["year"].min(), "-", gbd_raw["year"].max())
print("Age groups:")
print(sorted(gbd_raw["age_name"].unique().tolist()))

print("\nNulls in key columns:")
print(gbd_raw[key_cols].isna().sum())

dup_count = gbd_raw.duplicated(subset=["location_name", "sex_name", "age_name", "year"]).sum()
print("\nDuplicate rows by location/sex/age/year:", dup_count)

Countries: 38
Sex categories: ['Hombre', 'Mujer']
Years: 1990 - 2023
Age groups:
['10-14 años', '15-19 años', '20-24 años', '25-29 años', '30-34 años', '35-39 años', '40-44 años', '45-49 años', '5-9 años', '50-54 años', '55-59 años', '60-64 años', '65-69 años', '70+ años', 'Menores de 5 años']

Nulls in key columns:
location_name    0
sex_name         0
age_name         0
year             0
val              0
dtype: int64

Duplicate rows by location/sex/age/year: 0


## GBD download parameters

The dataset was downloaded manually from the IHME Global Burden of Disease (GBD) Results Tool.

Source:
https://vizhub.healthdata.org/gbd-results/

The OECD cohort used for the manual location selection is based on the official OECD members list:
https://www.oecd.org/about/members-and-partners/

Parameters used in the download:

Measure
- Prevalencia

Metric
- Tasa

Cause
- Trastornos del espectro autista

Sex
- Hombre
- Mujer

Age groups
- Menores de 5 años
- 5-9 años
- 10-14 años
- 15-19 años
- 20-24 años
- 25-29 años
- 30-34 años
- 35-39 años
- 40-44 años
- 45-49 años
- 50-54 años
- 55-59 años
- 60-64 años
- 65-69 años
- 70+ años

Years
- 1990-2023

Locations
- 38 OECD member countries, selected manually in the GBD interface

Downloaded file stored at:

`data/1_raw/ihme(gbd)_asd_prevalence_oecd_1990_2023.csv`